# Step 1: download the data

Downloads the Braun et al. 2023 first-trimester brain atlas (12 GB h5ad) if it is not there yet.

Source: Braun et al. 2023, *Science*, doi:10.1126/science.adf1226. The download link is the one given in the
authors' repository (github.com/linnarsson-lab/developing-human-brain) and is set in `config.yaml`.

## Setup

In [ ]:
# The notebooks live in notebooks/. Move to the repository root so config.yaml, data/ and results/
# are found, and make src/common.py importable.
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

from pathlib import Path

import requests
import yaml

cfg = yaml.safe_load(open("config.yaml"))["braun"]
h5ad = Path(cfg["file"])

## Download

In [ ]:
if h5ad.exists():
    # Nothing to do: running the pipeline again should not download 12 GB a second time
    print(f"{h5ad} already downloaded")
else:
    h5ad.parent.mkdir(parents=True, exist_ok=True)
    # Write to a temporary .part file first and rename only at the end. If the download breaks
    # halfway, no incomplete h5ad is left behind that later steps would mistake for the real file.
    part = h5ad.with_name(h5ad.name + ".part")
    with requests.get(cfg["url"], stream=True, timeout=60) as r:
        r.raise_for_status()  # stop with an error if the link no longer works
        with open(part, "wb") as f:
            # Stream in 8 MB pieces so the whole file never has to fit in memory
            for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                f.write(chunk)
    part.replace(h5ad)
    print(f"saved {h5ad}")